In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('training_data_hieu_khai_niem.csv')

In [3]:
df.head()

,question_id,subject,concept,ground_truth,student_answer,label
0,1,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung
1,2,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung
2,3,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet
3,4,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet
4,5,Vật lý 10,Định luật 1 Newton (quán tính),Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai


In [4]:
features = ['ground_truth', 'student_answer', 'label']
df = df[features]

In [5]:
from pyvi import ViTokenizer
def tokenize(st: str) -> list:
    token = ViTokenizer.tokenize(st).split()
    return token


In [6]:

df['tokenize_ground_truth'] = df['ground_truth'].apply(lambda st: tokenize(st))

In [7]:
df['tokenize_student_answer'] = df['student_answer'].apply(lambda st: tokenize(st))

In [8]:
df.head()

,ground_truth,student_answer,label,tokenize_ground_truth,tokenize_student_answer
0,Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Vật, đang, đứng, yên, hay, đang, chạy, đều, t..."
1,Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Em, nghĩ, là, khi, hợp_lực, tác_dụng, lên, vậ..."
2,Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Nếu, một, vật, không, chịu, tác_dụng, của, lự..."
3,Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Một, vật, không, chịu, lực, nào, tác_dụng, ho..."
4,Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Định_luật, 1, Newton, nói, là, vật, nào, cũng..."


In [9]:
from collections import Counter

def ngram_overlap(tokenize_ground_truth: list, tokenize_student_answer: list) -> float:
    bigram_gt = list(zip(tokenize_ground_truth, tokenize_ground_truth[1:]))
    trigram_gt = list(zip(tokenize_ground_truth, tokenize_ground_truth[1:], tokenize_ground_truth[2:]))
    
    bigram_sa = list(zip(tokenize_student_answer, tokenize_student_answer[1:]))
    trigram_sa = list(zip(tokenize_student_answer, tokenize_student_answer[1:], tokenize_student_answer[2:]))
    if not bigram_gt and not bigram_sa:
        return 0.0
    bigram_gtc = Counter(bigram_gt)
    trigram_gtc = Counter(trigram_gt)

    bigram_sac = Counter(bigram_sa)
    trigram_sac = Counter(trigram_sa)

    bigram_overlap = 0
    trigram_overlap = 0
    bigram_union = sum((bigram_gtc | bigram_sac).values())

    if bigram_union > 0:
        bigram_overlap = sum((bigram_gtc & bigram_sac).values()) / bigram_union

    trigram_union = sum((trigram_gtc | trigram_sac).values())
    if trigram_union > 0:
        trigram_overlap = sum((trigram_gtc & trigram_sac).values()) / trigram_union

    return (bigram_overlap + trigram_overlap) / 2


In [10]:
df['ngram_overlap'] = df.apply(lambda x: ngram_overlap(x['tokenize_ground_truth'], x['tokenize_student_answer']), axis=1)

In [11]:
df.head()

,ground_truth,student_answer,label,tokenize_ground_truth,tokenize_student_answer,ngram_overlap
0,Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Vật, đang, đứng, yên, hay, đang, chạy, đều, t...",0.029412
1,Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Em, nghĩ, là, khi, hợp_lực, tác_dụng, lên, vậ...",0.014925
2,Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Nếu, một, vật, không, chịu, tác_dụng, của, lự...",0.602124
3,Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Một, vật, không, chịu, lực, nào, tác_dụng, ho...",0.276423
4,Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Định_luật, 1, Newton, nói, là, vật, nào, cũng...",0.009434


In [12]:
def lcs(a: list, b: list) -> int:
    n, m = len(a), len(b)
    a, b = [0] + a, [0] + b
    pre = [0] * (m+5)
    cur = [0] * (m+5)
    for i in range(1, n+1):
        for j in range(1, m+1):
            if a[i] == b[j]:
                cur[j] = pre[j-1] + 1
            else:
                cur[j] = max(pre[j], cur[j-1])

        pre = cur[:]
    return cur[m]


In [13]:
df['lcs_ratio'] = df.apply(lambda x: (lcs(x['tokenize_ground_truth'], x['tokenize_student_answer']) / len(x['tokenize_ground_truth'])), axis=1)

In [14]:
df.head()

,ground_truth,student_answer,label,tokenize_ground_truth,tokenize_student_answer,ngram_overlap,lcs_ratio
0,Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Vật, đang, đứng, yên, hay, đang, chạy, đều, t...",0.029412,0.233333
1,Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Em, nghĩ, là, khi, hợp_lực, tác_dụng, lên, vậ...",0.014925,0.300000
2,Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Nếu, một, vật, không, chịu, tác_dụng, của, lự...",0.602124,0.900000
3,Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Một, vật, không, chịu, lực, nào, tác_dụng, ho...",0.276423,0.633333
4,Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Định_luật, 1, Newton, nói, là, vật, nào, cũng...",0.009434,0.233333


In [15]:
df['length_ratio'] = df.apply(lambda x: len(x['tokenize_student_answer']) / len(x['tokenize_ground_truth']), axis=1)

In [16]:
df.head()

,ground_truth,student_answer,label,tokenize_ground_truth,tokenize_student_answer,ngram_overlap,lcs_ratio,length_ratio
0,Nếu một vật không chịu tác dụng của lực nào ho...,Vật đang đứng yên hay đang chạy đều thì cứ giữ...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Vật, đang, đứng, yên, hay, đang, chạy, đều, t...",0.029412,0.233333,0.866667
1,Nếu một vật không chịu tác dụng của lực nào ho...,Em nghĩ là khi hợp lực tác dụng lên vật bằng 0...,dung,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Em, nghĩ, là, khi, hợp_lực, tác_dụng, lên, vậ...",0.014925,0.300000,1.366667
2,Nếu một vật không chịu tác dụng của lực nào ho...,Nếu một vật không chịu tác dụng của lực nào ho...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Nếu, một, vật, không, chịu, tác_dụng, của, lự...",0.602124,0.900000,0.966667
3,Nếu một vật không chịu tác dụng của lực nào ho...,Một vật không chịu lực nào tác dụng hoặc hợp l...,hoc_vet,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Một, vật, không, chịu, lực, nào, tác_dụng, ho...",0.276423,0.633333,0.800000
4,Nếu một vật không chịu tác dụng của lực nào ho...,Định luật 1 Newton nói là vật nào cũng cần có ...,sai,"[Nếu, một, vật, không, chịu, tác_dụng, của, lự...","[Định_luật, 1, Newton, nói, là, vật, nào, cũng...",0.009434,0.233333,0.866667


In [17]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    dot = np.dot(a, b)
    A = np.linalg.norm(a)
    B = np.linalg.norm(b)
    if A == 0 or B == 0:
        return .0
    return dot / (A * B)

In [18]:
from sentence_transformers import SentenceTransformer

def semantic_score(model, ground_truths: list, student_answers: list, batch_size=32) -> float:
    gt_embs = model.encode(ground_truths, batch_size=batch_size)
    st_embs = model.encode(student_answers, batch_size=batch_size)
    return [cosine_sim(gt, st) for gt, st in zip(gt_embs, st_embs)]

/workspaces/RealLearn-AITrain/reallearn/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [ ]:
import gc

MODELS = {
    'multilingual_mpnet': 'paraphrase-multilingual-mpnet-base-v2',
    'multilingual_minilm': 'paraphrase-multilingual-MiniLM-L12-v2',
    'vi_sbert': 'keepitreal/vietnamese-sbert',
}

for col_name, model_name in MODELS.items():
    model = SentenceTransformer(model_name)
    df[f'score_{col_name}'] = semantic_score(
        model, 
        df['ground_truth'].to_list(), 
        df['student_answer'].to_list()
    )
    del model
    gc.collect()

/workspaces/RealLearn-AITrain/reallearn/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/workspaces/RealLearn-AITrain/reallearn/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/workspaces/RealLearn-AITrain/reallearn/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` w